In [11]:
# Read bookings
booking_df = spark.read.parquet("gs://hilton-gcs-landing-varun/bookings/*")
booking_df.show(5)



+-----------+------------+------------+-------------+
|PROPERTY_ID|BOOKING_DATE|IS_AVAILABLE|PRICE_NUMERIC|
+-----------+------------+------------+-------------+
|   29864272|  2019-06-08|       false|         NULL|
|   29864272|  2019-06-01|       false|         NULL|
|   29864272|  2019-05-25|       false|         NULL|
|   29864272|  2019-05-18|       false|         NULL|
|   29864272|  2019-05-11|       false|         NULL|
+-----------+------------+------------+-------------+
only showing top 5 rows



In [12]:
# Read property
property_df = spark.read.parquet("gs://hilton-gcs-landing-varun/properties/*")
property_df.show(5)

+-----------+--------------------+-------------------+-------------+----------+--------------+------------+--------------+---------------+--------------------+-----------------------+-----------------------+-------------------------+---------------------------+-----------------------+-------------------------+-------------------------------+-----------------------+-------------------------+---------------------------+---------------+----------------------+----------------------------+-------+------------+-------+--------+---------+-----------------+-------------+---------------+------------+---------+--------+----+-------------+--------------------+-----------+-----+------------+-------------+----------------+------------+---------------+------------+--------------+--------------+----------------+----------------+---------------+---------------+---------------+----------------+-----------------+------------+-----------+--------------------+----------------------+-----------------------

In [13]:
# Read feedback
feedback_df = spark.read.parquet("gs://hilton-gcs-landing-varun/feedback/*")
feedback_df.show(5)

+-----------+---------+-----------+-----------+-------------+--------------------+-------------+
|PROPERTY_ID|REVIEW_ID|REVIEW_DATE|REVIEWER_ID|REVIEWER_NAME|            COMMENTS|REVIEW_LENGTH|
+-----------+---------+-----------+-----------+-------------+--------------------+-------------+
|    1177924|160691021| 2017-06-15|    8095518|       Edison|Doerthe’s apartme...|          146|
|    1177924|165823292| 2017-07-02|  112768978|    Stephanie|This was our firs...|          328|
|    1177924|168723814| 2017-07-10|     572790|       Cassie|Wonderful stay.  ...|           25|
|    1177924|171338880| 2017-07-17|   10546947|    Catherine|Doerthe was super...|           86|
|    1177924|172254290| 2017-07-20|   91444058|          Sam|Doerthe was great...|          136|
+-----------+---------+-----------+-----------+-------------+--------------------+-------------+
only showing top 5 rows



In [14]:
booking_df = booking_df.dropDuplicates()


In [15]:
median_price = booking_df.approxQuantile("PRICE_NUMERIC", [0.5], 0.01)[0]

In [16]:
median_price

60.0

In [17]:
booking_df = booking_df.fillna({"PRICE_NUMERIC": median_price})

In [18]:
from pyspark.sql.functions import col, count, when

null_counts = booking_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in booking_df.columns
])

null_counts.show()

+-----------+------------+------------+-------------+
|PROPERTY_ID|BOOKING_DATE|IS_AVAILABLE|PRICE_NUMERIC|
+-----------+------------+------------+-------------+
|          0|           0|           0|            0|
+-----------+------------+------------+-------------+



In [19]:
from pyspark.sql.functions import col, count, when

null_counts = feedback_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in feedback_df.columns
])

null_counts.show()

+-----------+---------+-----------+-----------+-------------+--------+-------------+
|PROPERTY_ID|REVIEW_ID|REVIEW_DATE|REVIEWER_ID|REVIEWER_NAME|COMMENTS|REVIEW_LENGTH|
+-----------+---------+-----------+-----------+-------------+--------+-------------+
|          0|        0|          0|          0|            0|     579|          579|
+-----------+---------+-----------+-----------+-------------+--------+-------------+



In [20]:
total_rows = feedback_df.count()

null_percentages = feedback_df.select([
    (count(when(col(c).isNull(), c)) / total_rows * 100).alias(c)
    for c in feedback_df.columns
])

null_percentages.show()

+-----------+---------+-----------+-----------+-------------+-------------------+-------------------+
|PROPERTY_ID|REVIEW_ID|REVIEW_DATE|REVIEWER_ID|REVIEWER_NAME|           COMMENTS|      REVIEW_LENGTH|
+-----------+---------+-----------+-----------+-------------+-------------------+-------------------+
|        0.0|      0.0|        0.0|        0.0|          0.0|0.14404454185625898|0.14404454185625898|
+-----------+---------+-----------+-----------+-------------+-------------------+-------------------+



In [21]:
feedback_df.count()

401959

In [22]:
feedback_df = feedback_df.dropna(subset=["COMMENTS"])

In [23]:
from pyspark.sql.functions import count, when, col

feedback_df.select(
    count(when(col("COMMENTS").isNull(), True)).alias("null_comments_after")
).show()

+-------------------+
|null_comments_after|
+-------------------+
|                  0|
+-------------------+



In [24]:
feedback_df.select(
    count(when(col("review_length").isNull(), True)).alias("null_review_length_after")
).show()

+------------------------+
|null_review_length_after|
+------------------------+
|                       0|
+------------------------+



In [25]:
from pyspark.sql.functions import col, count, when

null_counts = feedback_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in feedback_df.columns
])

null_counts.show()

+-----------+---------+-----------+-----------+-------------+--------+-------------+
|PROPERTY_ID|REVIEW_ID|REVIEW_DATE|REVIEWER_ID|REVIEWER_NAME|COMMENTS|REVIEW_LENGTH|
+-----------+---------+-----------+-----------+-------------+--------+-------------+
|          0|        0|          0|          0|            0|       0|            0|
+-----------+---------+-----------+-----------+-------------+--------+-------------+



In [26]:
from pyspark.sql.functions import col, count, when

property_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in property_df.columns
]).show()

+-----------+----+-------------------+-------------+----------+--------------+------------+--------------+---------------+------------------+-----------------------+-----------------------+-------------------------+---------------------------+-----------------------+-------------------------+-------------------------------+-----------------------+-------------------------+---------------------------+-------------+----------------------+----------------------------+-------+------------+-------+--------+---------+-----------------+-------------+---------+------------+---------+--------+----+--------+---------+-----------+-----+------------+-------------+----------------+------------+---------------+------------+--------------+--------------+----------------+----------------+---------------+---------------+---------------+----------------+-----------------+------------+-----------+--------------------+----------------------+-------------------------+---------------------+-----------------

In [27]:
property_df.count()

22552

In [28]:
property_df.printSchema()

root
 |-- PROPERTY_ID: string (nullable = true)
 |-- NAME: string (nullable = true)
 |-- EXPERIENCES_OFFERED: string (nullable = true)
 |-- THUMBNAIL_URL: string (nullable = true)
 |-- MEDIUM_URL: string (nullable = true)
 |-- XL_PICTURE_URL: string (nullable = true)
 |-- ATTENDENT_ID: string (nullable = true)
 |-- ATTENDENT_NAME: string (nullable = true)
 |-- ATTENDENT_SINCE: date (nullable = true)
 |-- ATTENDENT_LOCATION: string (nullable = true)
 |-- ATTENDENT_RESPONSE_TIME: string (nullable = true)
 |-- ATTENDENT_RESPONSE_RATE: decimal(38,6) (nullable = true)
 |-- ATTENDENT_ACCEPTANCE_RATE: decimal(38,6) (nullable = true)
 |-- ATTENDENT_IS_SUPERATTENDENT: boolean (nullable = true)
 |-- ATTENDENT_NEIGHBOURHOOD: string (nullable = true)
 |-- ATTENDENT_PROPERTYS_COUNT: decimal(38,0) (nullable = true)
 |-- ATTENDENT_TOTAL_PROPERTYS_COUNT: decimal(38,0) (nullable = true)
 |-- ATTENDENT_VERIFICATIONS: string (nullable = true)
 |-- ATTENDENT_HAS_PROFILE_PIC: boolean (nullable = true)
 |--

In [31]:
property_df = property_df.drop(
    "THUMBNAIL_URL",
    "MEDIUM_URL",
    "XL_PICTURE_URL",
)

In [32]:
property_df = property_df.drop(
    "ATTENDENT_ACCEPTANCE_RATE"
)

In [35]:
from pyspark.sql.functions import col, count, when

property_df.select(
    count(when(col("PRICE").isNull(), True)).alias("count")
).show()

+-----+
|count|
+-----+
|    0|
+-----+



In [40]:
property_df = property_df.drop(
    "LICENSE",
    "REQUIRES_LICENSE",
)

In [38]:
property_df.count()

22552

In [41]:
from pyspark.sql.functions import col

dim_property = property_df.select(
    "PROPERTY_ID",
    "PROPERTY_TYPE",
    "ROOM_TYPE",
    "ACCOMMODATES",
    "PRICE",
    "AVAILABILITY_30",
    "AVAILABILITY_365",
    "COUNTRY",
    "NEIGHBOURHOOD",
    "REVIEW_SCORES_RATING",
    "NUMBER_OF_REVIEWS",
    "REVIEWS_PER_MONTH"
)

In [42]:
dim_property = dim_property.withColumn(
    "occupancy_rate_30",
    (30 - col("AVAILABILITY_30")) / 30
)

In [45]:
dim_attendent = property_df.select(
    "ATTENDENT_ID",
    "ATTENDENT_RESPONSE_TIME",
    "ATTENDENT_RESPONSE_RATE",
).dropDuplicates(["ATTENDENT_ID"])

In [46]:
fact_bookings = booking_df.select(
    "PROPERTY_ID",
    "BOOKING_DATE",
    "PRICE_NUMERIC"
)

In [47]:
fact_bookings = fact_bookings.withColumn(
    "booking_revenue",
    col("PRICE_NUMERIC")
)

In [48]:
fact_reviews = feedback_df.select(
    "PROPERTY_ID",
    "REVIEW_DATE",
    "REVIEW_ID"
)

In [49]:
from pyspark.sql.functions import year, month, quarter, dayofmonth

dim_date = fact_bookings.select("BOOKING_DATE").dropDuplicates()

dim_date = dim_date.withColumn("year", year(col("BOOKING_DATE"))) \
                   .withColumn("month", month(col("BOOKING_DATE"))) \
                   .withColumn("quarter", quarter(col("BOOKING_DATE"))) \
                   .withColumn("day", dayofmonth(col("BOOKING_DATE")))

In [50]:
dim_property.write.mode("overwrite").parquet("gs://hilton-gcs-landing-varun/curated/dim_property/")
dim_attendent.write.mode("overwrite").parquet("gs://hilton-gcs-landing-varun/curated/dim_attendent/")
fact_bookings.write.mode("overwrite").parquet("gs://hilton-gcs-landing-varun/curated/fact_bookings/")
fact_reviews.write.mode("overwrite").parquet("gs://hilton-gcs-landing-varun/curated/fact_reviews/")
dim_date.write.mode("overwrite").parquet("gs://hilton-gcs-landing-varun/curated/dim_date/")

26/03/03 18:21:56 ERROR TransportClient: Failed to send RPC RPC 8476095777969410796 to /10.160.0.28:51770: io.netty.channel.StacklessClosedChannelException
io.netty.channel.StacklessClosedChannelException: null
	at io.netty.channel.AbstractChannel$AbstractUnsafe.write(Object, ChannelPromise)(Unknown Source) ~[netty-transport-4.1.100.Final.jar:4.1.100.Final]
26/03/03 18:21:56 WARN BlockManagerMasterEndpoint: Error trying to remove broadcast 99 from block manager BlockManagerId(15, cluster-1-w-0.asia-south1-a.c.sm-analytics-gcp-usecase.internal, 45779, None)
java.io.IOException: Failed to send RPC RPC 8476095777969410796 to /10.160.0.28:51770: io.netty.channel.StacklessClosedChannelException
	at org.apache.spark.network.client.TransportClient$RpcChannelListener.handleFailure(TransportClient.java:395) ~[spark-network-common_2.12-3.5.3.jar:3.5.3]
	at org.apache.spark.network.client.TransportClient$StdChannelListener.operationComplete(TransportClient.java:372) ~[spark-network-common_2.12-3.